# Anomaly Detection — Home Credit Application Data

**Week 2: Data Cleaning & Anomaly Detection Layer**

Continuation from the EDA notebook. Here we:
- Build manual red-flag features (extreme ratios, age vs years employed, etc.)
- Train an **Isolation Forest** for statistical anomaly detection
- Check whether "anomaly" correlates with default rate (an important insight for the report)
- Finalize via the `src/anomaly_detection.py` pipeline, save the model & data


## 1. Load Clean Data


In [1]:
import pandas as pd
import numpy as np

df_clean = pd.read_csv('../data/processed/application_train_clean.csv')
df_clean.shape

(307511, 79)

## 2. Feature Engineering: Manual Red Flags

Before feeding the model, first build "red flag" features that are easy to interpret (not just a black-box score):
1. Extreme credit/income ratio (top 1%)
2. Years employed that don't make sense relative to age (assumes minimum working age of 14)
3. Extreme income outliers (top 0.1%)
4. Total documents submitted
5. Credit bureau inquiries that are too frequent within a year (top 1%) — indication of "credit hunger"


In [2]:
def create_anomaly_features(df):
    df = df.copy()

    # 1. Extreme income-to-credit ratio (CREDIT_INCOME_RATIO already exists)
    # add the extreme flag
    df['FLAG_HIGH_CREDIT_INCOME_RATIO'] = (df['CREDIT_INCOME_RATIO'] > df['CREDIT_INCOME_RATIO'].quantile(0.99)).astype(int)

    # 2. Age vs years employed that doesn't make sense
    # DAYS_BIRTH is negative (days since birth), DAYS_EMPLOYED is negative (days since employment started)
    df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365
    df['EMPLOYED_YEARS'] = -df['DAYS_EMPLOYED'] / 365
    df['FLAG_EMPLOYED_LONGER_THAN_POSSIBLE'] = (
        df['EMPLOYED_YEARS'] > (df['AGE_YEARS'] - 14)  # assumes minimum working age of 14
    ).astype(int)

    # 3. Extreme income outlier
    df['FLAG_INCOME_OUTLIER'] = (df['AMT_INCOME_TOTAL'] > df['AMT_INCOME_TOTAL'].quantile(0.999)).astype(int)

    # 4. Inconsistent document count (e.g. submitting many documents but income doesn't match)
    doc_cols = [c for c in df.columns if c.startswith('FLAG_DOCUMENT_')]
    df['TOTAL_DOCUMENTS_SUBMITTED'] = df[doc_cols].sum(axis=1)

    # 5. Too many credit bureau inquiries in a short period (credit hunger)
    df['FLAG_HIGH_BUREAU_INQUIRY'] = (df['AMT_REQ_CREDIT_BUREAU_YEAR'] > df['AMT_REQ_CREDIT_BUREAU_YEAR'].quantile(0.99)).astype(int)

    return df

## 3. Check Flag Distribution


In [3]:
df_anomaly = create_anomaly_features(df_clean)

flag_cols = [c for c in df_anomaly.columns if c.startswith('FLAG_') and c not in df_clean.columns]
for col in flag_cols:
    print(col, df_anomaly[col].sum(), f"({df_anomaly[col].mean()*100:.2f}%)")

FLAG_HIGH_CREDIT_INCOME_RATIO 3076 (1.00%)
FLAG_EMPLOYED_LONGER_THAN_POSSIBLE 0 (0.00%)
FLAG_INCOME_OUTLIER 278 (0.09%)
FLAG_HIGH_BUREAU_INQUIRY 1237 (0.40%)


**Insight:** `FLAG_EMPLOYED_LONGER_THAN_POSSIBLE` = 0% — meaning there isn't a single application with a logically impossible age/years-employed combination. This is a good sign that the data is fairly consistent on this front, but it also means this rule isn't very useful as a discriminating feature (variance 0). A candidate to drop, or to relax the threshold for if it's meant to be kept.


## 4. Sanity Check: Age vs Years Employed


In [4]:
df_anomaly[['AGE_YEARS', 'EMPLOYED_YEARS']].describe()

,AGE_YEARS,EMPLOYED_YEARS
count,307511.000000,307511.000000
mean,43.936973,6.168784
std,11.956133,5.852585
min,20.517808,-0.000000
25%,34.008219,2.556164
50%,43.150685,4.515068
75%,53.923288,7.561644
max,69.120548,49.073973


**Insight:** the age range (20–69 years) and years employed (0–49 years) both make sense, no negative or oddly extreme values after the `DAYS_EMPLOYED` anomaly was handled in the previous notebook.


## 5. Isolation Forest — Exploration

Scale first (important since feature scales differ widely — income in the millions vs document count 0-20), then train an Isolation Forest with an initial assumption of ~2% anomalous data.


In [5]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Select the main numeric features for anomaly detection
# combination of original features + the red flag features we built
iso_features = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO',
    'AGE_YEARS', 'EMPLOYED_YEARS',
    'AMT_REQ_CREDIT_BUREAU_YEAR', 'TOTAL_DOCUMENTS_SUBMITTED'
]

X_iso = df_anomaly[iso_features].copy()

scaler = StandardScaler()
X_iso_scaled = scaler.fit_transform(X_iso)

# contamination = estimated proportion of anomalies, starting with an assumption of 2%
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42,
    n_jobs=-1
)

df_anomaly['ANOMALY_PRED'] = iso_forest.fit_predict(X_iso_scaled)
df_anomaly['ANOMALY_SCORE'] = iso_forest.decision_function(X_iso_scaled)

# -1 = anomaly, 1 = normal (sklearn's original output), convert to something more intuitive
df_anomaly['IS_ANOMALY'] = (df_anomaly['ANOMALY_PRED'] == -1).astype(int)

print(df_anomaly['IS_ANOMALY'].value_counts())
print(df_anomaly['IS_ANOMALY'].mean())

IS_ANOMALY
0    301360
1      6151
Name: count, dtype: int64
0.02000253649462946


## 6. Anomaly vs Default Rate

The core question here: is what's flagged as "anomaly" actually riskier?


In [6]:
print(df_anomaly.groupby('IS_ANOMALY')['TARGET'].mean())

IS_ANOMALY
0    0.081471
1    0.044383
Name: TARGET, dtype: float64


**Important insight (a caveat, not just a good number):** the default rate for the "anomaly" group is actually **lower** (4.4% vs 8.1%), the opposite of the common assumption that "anomaly = high risk". This makes sense once examined further in the next section — the Isolation Forest here is detecting *statistical outliers* (extreme values), not *suspicious behavioral outliers*. Important to document so it doesn't mislead the storytelling: "anomaly" ≠ automatically "fraud/high-risk".


## 7. Feature Characteristics: Anomaly vs Normal


In [7]:
# Look at feature statistics for the anomaly vs normal groups
for col in iso_features:
    print(col)
    print(df_anomaly.groupby('IS_ANOMALY')[col].mean())
    print('---')

AMT_INCOME_TOTAL
IS_ANOMALY
0    165882.732766
1    311623.575751
Name: AMT_INCOME_TOTAL, dtype: float64
---
AMT_CREDIT
IS_ANOMALY
0    5.776059e+05
1    1.648476e+06
Name: AMT_CREDIT, dtype: float64
---
AMT_ANNUITY
IS_ANOMALY
0    26466.538424
1    58559.939034
Name: AMT_ANNUITY, dtype: float64
---
AMT_GOODS_PRICE
IS_ANOMALY
0    5.179965e+05
1    1.533859e+06
Name: AMT_GOODS_PRICE, dtype: float64
---
CREDIT_INCOME_RATIO
IS_ANOMALY
0    3.840484
1    9.694046
Name: CREDIT_INCOME_RATIO, dtype: float64
---
ANNUITY_INCOME_RATIO
IS_ANOMALY
0    0.177854
1    0.331565
Name: ANNUITY_INCOME_RATIO, dtype: float64
---
AGE_YEARS
IS_ANOMALY
0    43.881604
1    46.649689
Name: AGE_YEARS, dtype: float64
---
EMPLOYED_YEARS
IS_ANOMALY
0    6.106552
1    9.217769
Name: EMPLOYED_YEARS, dtype: float64
---
AMT_REQ_CREDIT_BUREAU_YEAR
IS_ANOMALY
0    1.645597
1    1.538124
Name: AMT_REQ_CREDIT_BUREAU_YEAR, dtype: float64
---
TOTAL_DOCUMENTS_SUBMITTED
IS_ANOMALY
0    0.927356
1    1.067306
Name: TOTAL_DOCU

**Insight:** the "anomaly" group consistently has much higher income, credit, and annuity, plus higher age & years employed. This is a profile of **high-value / high-net-worth customers**, not fraudsters — which is why their default rate is actually lower (likely because they're more financially established). Conclusion: the Isolation Forest in this setup is better described as a "detector of statistically unusual profiles", and would need additional features (e.g. document consistency, data matching) to actually catch anomalies that are predictive of fraud/risk.


## 8. Finalization: Run via Pipeline Module

All the steps above (feature engineering + fit + apply Isolation Forest) have been consolidated into the `src/anomaly_detection.py` module, so it's reusable and applied consistently for train/test/OOT.


In [8]:
import sys
sys.path.append('../src')
from anomaly_detection import create_anomaly_features, fit_anomaly_detector, apply_anomaly_detector

df_anomaly = create_anomaly_features(df_clean)
iso_forest, scaler = fit_anomaly_detector(df_anomaly, contamination=0.02)
df_final = apply_anomaly_detector(df_anomaly, iso_forest, scaler)

df_final['IS_ANOMALY'].value_counts()

IS_ANOMALY
0    301360
1      6151
Name: count, dtype: int64

## 9. Save Model & Data


In [9]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(iso_forest, '../models/isolation_forest.pkl')
joblib.dump(scaler, '../models/anomaly_scaler.pkl')

df_final.to_csv('../data/processed/application_train_with_anomaly.csv', index=False)

---
### Summary & Next Steps
- ~2% of the data (6,151 rows) was flagged as anomalous by the Isolation Forest (matching the `contamination` setting).
- **Important caveat:** the anomaly group actually has a lower default rate (4.4% vs 8.1%) — they're a high-income/high-value profile, not fraud. This needs to be stated explicitly in the report so it doesn't mislead.
- `FLAG_EMPLOYED_LONGER_THAN_POSSIBLE` wasn't triggered at all (0%), a candidate to drop or relax the rule.
- The model (`isolation_forest.pkl`, `anomaly_scaler.pkl`) and data (`application_train_with_anomaly.csv`) have been saved.
- Next: **Week 3 — Multi-table feature engineering** (aggregating from `bureau`, `previous_application`, `installments_payments`, etc. to the `SK_ID_CURR` level).
